# Aula 04 - Notebook: Implementação de Conectivos Lógicos e Permissivos de Partida

Neste notebook implementamos as funções de avaliação lógica proposicional completas (AND, OR, NOT, XOR, IMPLICATION, BICONDITIONAL) e construímos os blocos de permissivos de partida (*Start Permissives*) e intertravamento contínuo para os atuadores da planta de fertilizantes.

In [ ]:
from typing import Dict
import pandas as pd
import itertools

# Operadores Fundamentais da Lógica Proposicional
def NOT(p: bool) -> bool:
    return not p

def AND(p: bool, q: bool) -> bool:
    return p and q

def OR(p: bool, q: bool) -> bool:
    return p or q

def XOR(p: bool, q: bool) -> bool:
    return p ^ q

def IMPLIES(p: bool, q: bool) -> bool:
    return (not p) or q

def IFF(p: bool, q: bool) -> bool:
    return p == q

print("Operadores lógicos proposicionais carregados com sucesso.")

## Bloco Lógico de Permissivo da Bomba P-101 (Ácido Fosfórico)

In [ ]:
def permissivo_bomba_P101(l_acid_low: bool, ls_suc_open: bool, p_discharge_high: bool, e1: bool, auto_mode: bool, manual_mode: bool) -> Dict[str, bool]:
    # Condição de modo exclusivo (Auto XOR Manual)
    modo_valido = XOR(auto_mode, manual_mode)
    
    # Condição combinada de permissivo
    permissivo = (NOT(l_acid_low) and 
                  ls_suc_open and 
                  NOT(p_discharge_high) and 
                  NOT(e1) and 
                  modo_valido)
    
    # Condição de trip imediato
    trip = l_acid_low or NOT(ls_suc_open) or p_discharge_high or e1
    
    return {
        'Permissivo_Habilitado': permissivo,
        'Trip_Ativo': trip,
        'Modo_Valido': modo_valido
    }

# Teste com diferentes cenários de campo
cenarios = [
    {"cenario": "Operação Normal (Auto)", "args": (False, True, False, False, True, False)},
    {"cenario": "Nível Baixo de Tanque", "args": (True, True, False, False, True, False)},
    {"cenario": "Válvula de Sucção Fechada", "args": (False, False, False, False, True, False)},
    {"cenario": "Sobrepressão na Descarga", "args": (False, True, True, False, True, False)},
    {"cenario": "Parada de Emergência Ativa", "args": (False, True, False, True, True, False)},
    {"cenario": "Conflito de Modo (Auto e Manual juntos)", "args": (False, True, False, False, True, True)},
]

resultados = []
for c in cenarios:
    res = permissivo_bomba_P101(*c["args"])
    resultados.append({
        "Cenário": c["cenario"],
        "Permissivo": res["Permissivo_Habilitado"],
        "Trip Ativo": res["Trip_Ativo"],
        "Modo Válido": res["Modo_Valido"]
    })

pd.DataFrame(resultados)

## Geração Automática de Tabela-Verdade para Validação Exhaustiva

In [ ]:
variaveis = ['l_acid_low', 'ls_suc_open', 'p_discharge_high', 'e1']
tabela = []

for combo in itertools.product([False, True], repeat=len(variaveis)):
    st = dict(zip(variaveis, combo))
    res = permissivo_bomba_P101(st['l_acid_low'], st['ls_suc_open'], st['p_discharge_high'], st['e1'], True, False)
    row = {**st, 'Permissivo': res['Permissivo_Habilitado'], 'Trip': res['Trip_Ativo']}
    tabela.append(row)

df_tv = pd.DataFrame(tabela)
print(f"Total de combinações avaliadas: {len(df_tv)}")
print(f"Combinações seguras que liberam a bomba: {df_tv['Permissivo'].sum()}")
print(df_tv.head(8))
